In [40]:
pip install bertopic

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [41]:
pip install vaderSentiment

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [42]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from bertopic import BERTopic
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [43]:
ai_df = pd.read_csv('AI.csv', encoding='latin1')
ai_df.rename(columns={'Number': 'id'}, inplace=True)
benefits_df = pd.read_csv('dat_benefits_text.csv', encoding='latin1')
risks_df = pd.read_csv('dat_risks_text.csv', encoding='latin1')

In [44]:
ai_df.head()

,id,risks_AI_avg,support_company,gender_bin,objective_threat,trait_risk,educ_short,percent_job_gain,age_group,education,country,manipulation_check2,weight
0,1,4.0,4.0,Men,NaN,certain,0,100%,45 to 64,HS or less,US,Pass,0.2703
1,2,4.0,2.0,NaN,0.200000,certain,1,30%,45 to 64,Postgrad,US,Pass,0.2709
2,3,3.0,4.0,Men,NaN,certain,0,70%,45 to 64,HS or less,US,Pass,0.6464
3,4,2.5,3.0,Men,0.466667,certain,0,100%,30 to 44,HS or less,US,Pass,2.7281
4,5,3.0,4.0,Men,NaN,certain,0,30%,65 or older,Some college,US,Fail,0.8643


In [45]:
benefits_df.head()

,id,benefits_text,gender_bin,gender_cat
0,1,Not sure,Man,Man
1,2,No I cannot,Man,Man
2,3,No idea,Woman,Woman
3,4,Efficency,Man,Man
4,5,In medicine,Woman,Woman


In [46]:
risks_df.head()

,id,risks_text,gender_bin,gender_cat
0,2,Not sure,Man,Man
1,3,Taking over the world,Man,Man
2,8,People losing jobs,Woman,Woman
3,10,Job loss,Man,Man
4,12,Crime,Woman,Woman


In [47]:
df = ai_df.merge(benefits_df, on="id", how="left")
df = df.merge(risks_df, on="id", how="left")

In [48]:
df.head()

,id,risks_AI_avg,support_company,gender_bin_x,objective_threat,trait_risk,educ_short,percent_job_gain,age_group,education,country,manipulation_check2,weight,benefits_text,gender_bin_y,gender_cat_x,risks_text,gender_bin,gender_cat_y
0,1,4.0,4.0,Men,NaN,certain,0,100%,45 to 64,HS or less,US,Pass,0.2703,Not sure,Man,Man,NaN,NaN,NaN
1,2,4.0,2.0,NaN,0.200000,certain,1,30%,45 to 64,Postgrad,US,Pass,0.2709,No I cannot,Man,Man,Not sure,Man,Man
2,3,3.0,4.0,Men,NaN,certain,0,70%,45 to 64,HS or less,US,Pass,0.6464,No idea,Woman,Woman,Taking over the world,Man,Man
3,4,2.5,3.0,Men,0.466667,certain,0,100%,30 to 44,HS or less,US,Pass,2.7281,Efficency,Man,Man,NaN,NaN,NaN
4,5,3.0,4.0,Men,NaN,certain,0,30%,65 or older,Some college,US,Fail,0.8643,In medicine,Woman,Woman,NaN,NaN,NaN


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3049 entries, 0 to 3048
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   3049 non-null   int64  
 1   risks_AI_avg         3049 non-null   float64
 2   support_company      2689 non-null   float64
 3   gender_bin_x         3007 non-null   object 
 4   objective_threat     1600 non-null   float64
 5   trait_risk           3049 non-null   object 
 6   educ_short           3049 non-null   int64  
 7   percent_job_gain     3049 non-null   object 
 8   age_group            3049 non-null   object 
 9   education            3049 non-null   object 
 10  country              3049 non-null   object 
 11  manipulation_check2  3045 non-null   object 
 12  weight               3049 non-null   float64
 13  benefits_text        2891 non-null   object 
 14  gender_bin_y         3001 non-null   object 
 15  gender_cat_x         3001 non-null   o

In [50]:
df.describe()

,id,risks_AI_avg,support_company,objective_threat,educ_short,weight
count,3049.000000,3049.000000,2689.000000,1600.000000,3049.000000,3049.000000
mean,1525.000000,4.653657,3.119747,0.518458,0.299442,0.999236
std,880.314811,2.508140,1.094399,0.218976,0.458089,0.616210
min,1.000000,0.000000,1.000000,0.000000,0.000000,0.137600
25%,763.000000,3.000000,2.000000,0.366667,0.000000,0.647400
50%,1525.000000,5.000000,3.000000,0.500000,0.000000,0.834400
75%,2287.000000,6.000000,4.000000,0.666667,1.000000,1.135600
max,3049.000000,10.000000,5.000000,1.000000,1.000000,5.739000


In [51]:
df["combined_text"] = df[[c for c in df.columns if c.endswith("text")]].fillna("").agg(" ".join, axis=1)

In [52]:
df.head()

,id,risks_AI_avg,support_company,gender_bin_x,objective_threat,trait_risk,educ_short,percent_job_gain,age_group,education,country,manipulation_check2,weight,benefits_text,gender_bin_y,gender_cat_x,risks_text,gender_bin,gender_cat_y,combined_text
0,1,4.0,4.0,Men,NaN,certain,0,100%,45 to 64,HS or less,US,Pass,0.2703,Not sure,Man,Man,NaN,NaN,NaN,Not sure
1,2,4.0,2.0,NaN,0.200000,certain,1,30%,45 to 64,Postgrad,US,Pass,0.2709,No I cannot,Man,Man,Not sure,Man,Man,No I cannot Not sure
2,3,3.0,4.0,Men,NaN,certain,0,70%,45 to 64,HS or less,US,Pass,0.6464,No idea,Woman,Woman,Taking over the world,Man,Man,No idea Taking over the world
3,4,2.5,3.0,Men,0.466667,certain,0,100%,30 to 44,HS or less,US,Pass,2.7281,Efficency,Man,Man,NaN,NaN,NaN,Efficency
4,5,3.0,4.0,Men,NaN,certain,0,30%,65 or older,Some college,US,Fail,0.8643,In medicine,Woman,Woman,NaN,NaN,NaN,In medicine


# 3. RoBERTa Embeddings (HuggingFace + PyTorch)

In [53]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModel.from_pretrained("roberta-base")
model.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [54]:
@torch.no_grad()
def get_embeddings(text_list, batch_size=16, max_length=128):
    all_emb = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=max_length)
        outputs = model(**inputs)
        # mean pooling
        emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        all_emb.append(emb)
    return np.vstack(all_emb)

print("Generating embeddings...")
embeddings = get_embeddings(df["combined_text"].tolist())

Generating embeddings...


# 4. BERTopic (Topic Proportions as Features)

In [55]:
topic_model = BERTopic(
    verbose=True,
    calculate_probabilities=True
)

topics, probs = topic_model.fit_transform(df["combined_text"], embeddings)

df["topic"] = topics
# df["is_outlier"] = (df["topic"] == -1).astype(int)

# Now this works
probs = np.array(probs)

probs_df = pd.DataFrame(
    probs,
    columns=[f"topic_{i}" for i in range(probs.shape[1])]
)

# probs_df = probs_df.div(probs_df.sum(axis=1).replace(0, 1), axis=0)

df = pd.concat([df.reset_index(drop=True), probs_df], axis=1)

# Inspect topics
print(topic_model.get_topic_info().head(10))
for tid in topic_model.get_topic_info()["Topic"].head(8):
    print(f"\nTopic {tid}:")
    print(topic_model.get_topic(tid))

2026-03-18 16:45:50,574 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-18 16:45:57,560 - BERTopic - Dimensionality - Completed ✓
2026-03-18 16:45:57,561 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-18 16:45:57,821 - BERTopic - Cluster - Completed ✓
2026-03-18 16:45:57,829 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-18 16:45:57,868 - BERTopic - Representation - Completed ✓


   Topic  Count                                   Name  \
0     -1    339                   -1_there_the_none_no   
1      0   1397                        0_and_the_of_ai   
2      1    238                    1_to_do_able_enough   
3      2    135                     2_dont_know_im_any   
4      3     64  3_speed_medical_healthcare_convenient   
5      4     60               4_ease_saves_work_faster   
6      5     57       5_efficiency_cost_operation_less   
7      6     48    6_productivity_makes_improving_life   
8      7     47                    7_idea_have_no_know   
9      8     41     8_easier_making_things_efficiently   

                                      Representation  \
0  [there, the, none, no, faster, for, are, is, a...   
1     [and, the, of, ai, it, to, in, for, can, jobs]   
2  [to, do, able, enough, people, being, make, it...   
3  [dont, know, im, any, about, not, there, takin...   
4  [speed, medical, healthcare, convenient, produ...   
5  [ease, saves, work, fa

# 5. Sentiment (VADER)

In [56]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0.0
    return analyzer.polarity_scores(text)["compound"]  # -1 to 1

print("Computing sentiment...")
df["sentiment"] = df["combined_text"].apply(get_sentiment)

Computing sentiment...


In [57]:
df.head()

,id,risks_AI_avg,support_company,gender_bin_x,objective_threat,trait_risk,educ_short,percent_job_gain,age_group,education,...,topic_31,topic_32,topic_33,topic_34,topic_35,topic_36,topic_37,topic_38,topic_39,sentiment
0,1,4.0,4.0,Men,NaN,certain,0,100%,45 to 64,HS or less,...,4.772585e-308,7.659696e-309,4.579036e-308,5.163023e-308,2.784349e-308,1.106384e-308,7.543628e-308,8.522082e-309,7.358054e-308,-0.2411
1,2,4.0,2.0,NaN,0.200000,certain,1,30%,45 to 64,Postgrad,...,2.434875e-02,3.379408e-03,2.168407e-02,4.267169e-02,1.855492e-02,5.289022e-03,1.640313e-02,4.050463e-03,2.685738e-02,-0.1250
2,3,3.0,4.0,Men,NaN,certain,0,70%,45 to 64,HS or less,...,2.514947e-02,3.658890e-03,2.281925e-02,4.353050e-02,2.080159e-02,5.788283e-03,1.646974e-02,4.304862e-03,2.692077e-02,-0.2960
3,4,2.5,3.0,Men,0.466667,certain,0,100%,30 to 44,HS or less,...,1.254601e-02,1.545812e-03,1.442863e-02,1.159352e-02,6.944757e-02,2.420494e-03,6.152697e-03,1.694968e-03,1.048419e-02,0.0000
4,5,3.0,4.0,Men,NaN,certain,0,30%,65 or older,Some college,...,4.595418e-308,6.767595e-309,5.217087e-308,4.478315e-308,1.746591e-307,1.052367e-308,2.474884e-308,7.401013e-309,3.957668e-308,0.0000


# 6. Uncertainty Score (0-5)
## Simple lexicon-based scoring

In [58]:
uncertain_terms = [
    r"\bmaybe\b", r"\bnot sure\b", r"\bunsure\b", r"\buncertain\b", r"\bi think\b",
    r"\bprobably\b", r"\bpossibly\b", r"\bcould\b", r"\bmight\b", r"\bperhaps\b",
    r"\bdon'?t know\b", r"\bno idea\b"
]
uncertain_regex = re.compile("|".join(uncertain_terms), flags=re.IGNORECASE)

def uncertainty_score(text, cap=5):
    if not isinstance(text, str) or text.strip() == "":
        return 0
    matches = re.findall(uncertain_regex, text)
    return int(min(len(matches), cap))

print("Computing uncertainty...")
df["uncertainty"] = df["combined_text"].apply(uncertainty_score)

Computing uncertainty...


In [59]:
df.head()

,id,risks_AI_avg,support_company,gender_bin_x,objective_threat,trait_risk,educ_short,percent_job_gain,age_group,education,...,topic_32,topic_33,topic_34,topic_35,topic_36,topic_37,topic_38,topic_39,sentiment,uncertainty
0,1,4.0,4.0,Men,NaN,certain,0,100%,45 to 64,HS or less,...,7.659696e-309,4.579036e-308,5.163023e-308,2.784349e-308,1.106384e-308,7.543628e-308,8.522082e-309,7.358054e-308,-0.2411,1
1,2,4.0,2.0,NaN,0.200000,certain,1,30%,45 to 64,Postgrad,...,3.379408e-03,2.168407e-02,4.267169e-02,1.855492e-02,5.289022e-03,1.640313e-02,4.050463e-03,2.685738e-02,-0.1250,1
2,3,3.0,4.0,Men,NaN,certain,0,70%,45 to 64,HS or less,...,3.658890e-03,2.281925e-02,4.353050e-02,2.080159e-02,5.788283e-03,1.646974e-02,4.304862e-03,2.692077e-02,-0.2960,1
3,4,2.5,3.0,Men,0.466667,certain,0,100%,30 to 44,HS or less,...,1.545812e-03,1.442863e-02,1.159352e-02,6.944757e-02,2.420494e-03,6.152697e-03,1.694968e-03,1.048419e-02,0.0000,0
4,5,3.0,4.0,Men,NaN,certain,0,30%,65 or older,Some college,...,6.767595e-309,5.217087e-308,4.478315e-308,1.746591e-307,1.052367e-308,2.474884e-308,7.401013e-309,3.957668e-308,0.0000,0


In [60]:
print(df.columns.tolist())

['id', 'risks_AI_avg', 'support_company', 'gender_bin_x', 'objective_threat', 'trait_risk', 'educ_short', 'percent_job_gain', 'age_group', 'education', 'country', 'manipulation_check2', 'weight', 'benefits_text', 'gender_bin_y', 'gender_cat_x', 'risks_text', 'gender_bin', 'gender_cat_y', 'combined_text', 'topic', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17', 'topic_18', 'topic_19', 'topic_20', 'topic_21', 'topic_22', 'topic_23', 'topic_24', 'topic_25', 'topic_26', 'topic_27', 'topic_28', 'topic_29', 'topic_30', 'topic_31', 'topic_32', 'topic_33', 'topic_34', 'topic_35', 'topic_36', 'topic_37', 'topic_38', 'topic_39', 'sentiment', 'uncertainty']


In [61]:
df = df.loc[:, ~df.columns.duplicated()]

In [63]:
df["gender_clean"] = df["gender_bin"]

# Fill missing from other sources
df["gender_clean"] = df["gender_clean"].fillna(df["gender_bin_x"])
df["gender_clean"] = df["gender_clean"].fillna(df["gender_bin_y"])

In [64]:
df["gender_clean"].isna().sum()

np.int64(0)

# 7. Build Feature Set

In [65]:
#structured = [c for c in [
#    "gender", "age", "education", "risk_orientation"
#] if c in df.columns]

structured = [
    "gender_clean",
    "age_group",
    "education",
    "trait_risk",
    "objective_threat",
    "percent_job_gain"
]

topic_cols = [c for c in df.columns if c.startswith("topic_")]
text_feats = topic_cols + ["sentiment", "uncertainty"]

# Targets
risk_target = "risks_AI_avg"            # continuous
support_target = "support_company"     # ordinal (1-5)

# Prepare modeling df
model_cols = structured + text_feats + [risk_target, support_target]
df_m = df[model_cols].dropna().copy()

# Basic encoding (example)
for col in structured:
    if df_m[col].dtype == "object":
        df_m[col] = df_m[col].astype("category").cat.codes

In [66]:
# 8. Model A: Text + Structured -> Risk Perception

In [67]:
X = df_m[structured + text_feats]
y = df_m[risk_target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("\nRisk Model Performance")
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

# Feature importance
feat_imp = pd.DataFrame({"feature": X.columns, "importance": rf.feature_importances_})\
    .sort_values("importance", ascending=False)
print("\nTop Features (Risk):")
print(feat_imp.head(20))


Risk Model Performance
MSE: 6.531324151234568
R2: 0.04761384397726687

Top Features (Risk):
             feature  importance
4   objective_threat    0.100706
46         sentiment    0.073680
1          age_group    0.046707
6            topic_0    0.035502
5   percent_job_gain    0.031839
0       gender_clean    0.029532
2          education    0.027447
9            topic_3    0.024846
7            topic_1    0.024515
16          topic_10    0.023971
8            topic_2    0.023061
40          topic_34    0.021751
3         trait_risk    0.020867
20          topic_14    0.020472
36          topic_30    0.019819
31          topic_25    0.018509
19          topic_13    0.018158
15           topic_9    0.017327
14           topic_8    0.017308
41          topic_35    0.017100


In [68]:
# 9. Model B: Risk -> AI Support (Ordinal via OLS approx)
# (For proper ordinal, use statsmodels OrderedModel)

In [69]:
X2 = df_m[[risk_target] + structured + text_feats]
y2 = df_m[support_target]

X2 = sm.add_constant(X2)
ols_support = sm.OLS(y2, X2).fit(cov_type='HC3')
print("\nSupport Model (OLS approx):")
print(ols_support.summary())


Support Model (OLS approx):
                            OLS Regression Results                            
Dep. Variable:        support_company   R-squared:                       0.140
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     4.762
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           2.14e-23
Time:                        16:53:07   Log-Likelihood:                -2041.3
No. Observations:                1436   AIC:                             4183.
Df Residuals:                    1386   BIC:                             4446.
Df Model:                          49                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const      

In [70]:
# 10. Mediation Analysis (Baron & Kenny style)
# Path: Text -> Risk -> Support

In [75]:
# Step 1: M ~ X (Risk on Text+Structured)
formula_m = f"{risk_target} ~ " + " + ".join(structured + text_feats)
model_m = smf.ols(formula_m, data=df_m).fit(cov_type='HC3')

# Step 2: Y ~ X (Support on Text+Structured)
formula_yx = f"{support_target} ~ " + " + ".join(structured + text_feats)
model_yx = smf.ols(formula_yx, data=df_m).fit(cov_type='HC3')

# Step 3: Y ~ M + X (Support on Risk + Text+Structured)
formula_y = f"{support_target} ~ {risk_target} + " + " + ".join(structured + text_feats)
model_y = smf.ols(formula_y, data=df_m).fit(cov_type='HC3')

print("\nMediation Results:")
print("\nModel M (Risk ~ X):\n", model_m.summary())
print("\nModel Y|X (Support ~ X):\n", model_yx.summary())
print("\nModel Y (Support ~ Risk + X):\n", model_y.summary())


Mediation Results:

Model M (Risk ~ X):
                             OLS Regression Results                            
Dep. Variable:           risks_AI_avg   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.088
Method:                 Least Squares   F-statistic:                     5.276
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           1.04e-26
Time:                        16:54:26   Log-Likelihood:                -3306.2
No. Observations:                1436   AIC:                             6710.
Df Residuals:                    1387   BIC:                             6969.
Df Model:                          48                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

In [76]:
def bootstrap_indirect_effect(df, x_cols, m_col, y_col, n_boot=500):
    inds = []
    for _ in range(n_boot):
        samp = df.sample(frac=1.0, replace=True)
        m = smf.ols(f"{m_col} ~ " + " + ".join(x_cols), data=samp).fit()
        y = smf.ols(f"{y_col} ~ {m_col} + " + " + ".join(x_cols), data=samp).fit()
        # aggregate indirect effect as mean over Xs: sum(a_i * b)
        b = y.params.get(m_col, 0)
        a = np.array([m.params.get(col, 0) for col in x_cols])
        inds.append((a * b).sum())
    return np.percentile(inds, [2.5, 50, 97.5])

ci = bootstrap_indirect_effect(df_m, structured + text_feats, risk_target, support_target)
print("\nBootstrap indirect effect CI (2.5%, 50%, 97.5%):", ci)


Bootstrap indirect effect CI (2.5%, 50%, 97.5%): [-6.14614707  0.57942611  4.7738177 ]


In [77]:
# 11. Research Question Checks
# Do language differences explain gender gaps?

In [78]:
# Compare gender coefficient with/without text features in Risk model
if "gender" in df_m.columns:
    base_cols = [c for c in structured if c != "gender"]
    m_base = smf.ols(f"{risk_target} ~ gender + " + " + ".join(base_cols), data=df_m).fit(cov_type='HC3')
    m_text = smf.ols(f"{risk_target} ~ gender + " + " + ".join(base_cols + text_feats), data=df_m).fit(cov_type='HC3')
    print("\nGender effect WITHOUT text features:", m_base.params.get("gender"))
    print("Gender effect WITH text features:", m_text.params.get("gender"))

In [79]:
df.to_csv("enhanced_with_topics_sentiment_uncertainty.csv", index=False)
feat_imp.to_csv("feature_importance_risk.csv", index=False)

print("\nPipeline complete.")


Pipeline complete.
